# HomeReach Melbourne — 10: Ridge regression versus the rental baselines

**Research only. The deployed explorer and its frozen data do not change.**

Notebook 09 established two reference forecasts. This notebook evaluates **one fixed Ridge specification** on exactly the same **589 LGA-quarter validation targets**. It does not search for a favourable specification after seeing the scores.

- Target: published nominal median weekly rent, AUD/week, two-bedroom flats.
- History: original rental series from 1999Q2; validation targets 2019Q1–2023Q3.
- Final evaluation reserved: 2023Q4–2025Q3, **not forecast or scored here**.
- One separate Ridge model per LGA. Four prior-quarter rents plus three known calendar indicators.
- Model: `StandardScaler` followed by `Ridge(alpha=1.0, fit_intercept=True, solver="svd")`.
- Expanding-window, repeated one-quarter forecasts; both scaler and estimator refitted at every origin.

**Before running:** keep the `feat/rental-forecast-baselines` branch active, select the existing `homereach` kernel and run the code cells from top to bottom. Notebook 09 must have completed; its saved results are inputs, not overwritten outputs. No new installations or raw GTFS scans are required.

The ZIP adds this notebook and `config/rental_ridge_protocol.json`. It does not replace `rental_forecast_protocol.json`. Save personal notes outside the generated findings report.

A successful checkpoint means the experiment followed its rules—not that Ridge beat a simple baseline.


## 1. Check the environment and unchanged rental source

This verifies the same 3,286-row rental table and source fingerprint used by notebook 09. Quarter-end dates are compared in a common nanosecond unit for pandas 2/3 compatibility. The complete input is checked structurally; final-period values will be removed before feature construction.

The branch guard prevents accidental work on deployed `main`. No files are downloaded and no packages are installed. The notebook uses the existing `homereach.io.dffh_rent.read_panel_csv()` helper, not a separately retyped rental table.


In [1]:
print("Starting notebook 10 — Ridge validation...", flush=True)
from pathlib import Path
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
import hashlib
import json
import math
import subprocess
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly
from IPython.display import display
from homereach.io.dffh_rent import COLUMN_ORDER, read_panel_csv


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def read_json(path):
    value = json.loads(Path(path).read_text(encoding="utf-8-sig"))
    require(isinstance(value, dict), f"Expected a JSON object: {path}")
    return value


ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "pyproject.toml").is_file() and (p / "data").is_dir()), None)
require(ROOT is not None, "Open this notebook from the existing HomeReach project.")
branch_result = subprocess.run(
    ["git", "-C", str(ROOT), "branch", "--show-current"],
    capture_output=True, text=True, timeout=10, check=True,
)
BRANCH = branch_result.stdout.strip()
require(BRANCH == "feat/rental-forecast-baselines",
        "Use the research branch feat/rental-forecast-baselines before running notebook 10; do not develop on deployed main.")

INPUT = ROOT / "data/processed/rent_quarterly.csv"
QUALITY_PATH = ROOT / "data/processed/rent_quality_report.json"
SPEC_PATH = ROOT / "config/rent_snapshot.json"
PROTOCOL_PATH = ROOT / "config/rental_forecast_protocol.json"
for path in [INPUT, QUALITY_PATH, SPEC_PATH, PROTOCOL_PATH]:
    require(path.is_file(), f"Missing required file: {path.relative_to(ROOT)}")

spec = read_json(SPEC_PATH)
quality_report = read_json(QUALITY_PATH)
protocol = read_json(PROTOCOL_PATH)
EXPECTED_SOURCE_SHA256 = "b8e96347494f09fea2a2ace55997fd34c9398f8509cb3450d65444aebfaaa837"
require(spec["sha256"] == protocol["source_sha256"] == EXPECTED_SOURCE_SHA256,
        "This notebook requires the audited 2025Q3 rental snapshot. Do not change fingerprints to bypass it.")
protected_paths = [INPUT, QUALITY_PATH, SPEC_PATH, PROTOCOL_PATH]
raw_rent = ROOT / "data/raw/rent" / spec["filename"]
if raw_rent.is_file():
    require(sha256_file(raw_rent) == EXPECTED_SOURCE_SHA256, "The original rental workbook changed.")
    protected_paths.append(raw_rent)
for relative in ["data/processed/rent_quarterly.parquet", "docs/data_registry.csv", "README.md"]:
    if (ROOT / relative).is_file():
        protected_paths.append(ROOT / relative)
for directory in ["app", "deploy"]:
    protected_paths.extend(p for p in (ROOT / directory).rglob("*")
                           if p.is_file() and "__pycache__" not in p.parts and p.suffix != ".pyc")
input_hashes = {p: sha256_file(p) for p in sorted(set(protected_paths))}
print("Research branch:", BRANCH)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Protected existing files:", len(input_hashes))

require(quality_report.get("source", {}).get("sha256") == EXPECTED_SOURCE_SHA256,
        "Rental quality report belongs to another workbook.")
require(quality_report.get("output_sha256", {}).get(INPUT.name) == input_hashes[INPUT],
        "Rental CSV differs from its pipeline report. Investigate; do not edit the hash.")
require(quality_report.get("csv_readback_matches") is True,
        "The rental pipeline has not recorded a successful CSV read-back.")
q = quality_report.get("quality", {})
require(q.get("rows") == 3286 and q.get("lga_count") == 31 and q.get("source_values_verified") == 6572,
        "Unexpected rental quality checkpoint.")
panel = read_panel_csv(INPUT)
require(panel.columns.tolist() == COLUMN_ORDER, "The 14-column rental schema changed.")
require(len(panel) == 3286 and panel.rental_area_name.nunique() == 31, "Unexpected panel size.")
require(not panel.duplicated(["rental_area_name", "property_type", "quarter"]).any(), "Duplicate rental key.")
for column, expected in {
    "property_type": "2 bedroom flats", "area_type": "LGA", "source_file": spec["filename"],
    "source_sheet": spec["sheet"], "source_reporting_quarter": "2025Q3",
}.items():
    require(panel[column].eq(expected).all(), f"Unexpected source scope: {column}")
for column in ["median_weekly_rent", "published_count"]:
    require(panel[column].notna().all() and np.isfinite(panel[column].astype(float)).all()
            and panel[column].gt(0).all(), f"Invalid numeric input: {column}")
periods = pd.PeriodIndex(panel.quarter, freq="Q-DEC")
expected_dates = periods.to_timestamp(how="end").normalize().as_unit("ns")
actual_dates = pd.DatetimeIndex(panel.quarter_end).as_unit("ns")
require(np.array_equal(actual_dates.asi8, expected_dates.asi8), "Quarter labels and dates disagree.")
ALL_QUARTERS = pd.period_range("1999Q2", "2025Q3", freq="Q-DEC")
for _, group in panel.groupby(["rental_area_name", "property_type"], sort=False):
    require(sorted(group.quarter.tolist()) == ALL_QUARTERS.astype(str).tolist(), "Missing or altered quarters.")
require(panel.source_median_cell.str.fullmatch(r"[A-Z]+[1-9][0-9]*").all(), "Missing source cell references.")
print("RENTAL INPUT CHECKPOINT PASSED")
print("Source rows: 3286 | Columns: 14 | LGAs: 31 | Quarters: 106")
print("Source CSV matches its local pipeline fingerprint: True")
print("Transport data loaded: False | Holdout forecasts scored: 0")
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
print("scikit-learn:",sklearn.__version__)


Starting notebook 10 — Ridge validation...
Research branch: feat/rental-forecast-baselines
Python: 3.12.14 | pandas: 3.0.3
Protected existing files: 26
RENTAL INPUT CHECKPOINT PASSED
Source rows: 3286 | Columns: 14 | LGAs: 31 | Quarters: 106
Source CSV matches its local pipeline fingerprint: True
Transport data loaded: False | Holdout forecasts scored: 0
scikit-learn: 1.8.0


## 2. Freeze the new specification and verify notebook 09

The previous baseline protocol and its manifest must match. This notebook also rechecks the saved baseline predictions against their local hashes. It uses the original reporting periods, target and aggregation policy.

**New design choices, fixed for this experiment:**

| Component | Decision |
|---|---|
| Training scope | A separate model for each LGA/category; no cross-area pooling |
| Rental inputs | `rent_lag_1`, `rent_lag_2`, `rent_lag_3`, `rent_lag_4` |
| Calendar inputs | `target_is_q2`, `target_is_q3`, `target_is_q4`; Q1 is the reference |
| Regularisation | Fixed alpha = 1.0, no search or tuning |
| Target transform | None: predict the published weekly rent directly |
| Training window | All earlier eligible examples; refit at each forecast origin |
| Preprocessing | Mean/scale learned only from that fit's training examples |

For example, a forecast for **2019Q1** uses rental inputs from **2018Q4, 2018Q3, 2018Q2 and 2018Q1**. The calendar indicators are all zero because the target is Q1. For a Q2 target they are `[1, 0, 0]`. The future quarter's calendar is already known; its rent is not.

The four warm-up quarters cannot form a complete feature vector. Consequently the first supervised training target is **2000Q2**, not 1999Q2. For the first validation forecast there are **75 training examples per LGA**, growing to **93** at the last origin. This does not change the 589 validation targets used for comparison.

The first candidate is deliberately fixed rather than claiming alpha=1 is optimal. A poor result remains a legitimate result. Any later alternative needs a separately recorded experiment.


In [2]:
# The original protocol is immutable; this is a separate model specification.
RIDGE_PROTOCOL_PATH = ROOT / 'config/rental_ridge_protocol.json'
BASE_DIR = ROOT / 'data/interim/rental_forecast_baselines'
BASE_MANIFEST_PATH = BASE_DIR / 'baseline_manifest.json'
for p in [RIDGE_PROTOCOL_PATH, BASE_MANIFEST_PATH]:
    require(p.is_file(), f'Missing required input: {p.relative_to(ROOT)}')
ridge_protocol = read_json(RIDGE_PROTOCOL_PATH)
baseline_manifest = read_json(BASE_MANIFEST_PATH)

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

BASE_PROTOCOL_HASH = canonical_hash(protocol)
require(BASE_PROTOCOL_HASH == ridge_protocol['baseline_protocol_sha256'],
        'The baseline protocol differs from the reviewed design; do not edit it to bypass this check.')
require(baseline_manifest['protocol_sha256'] == BASE_PROTOCOL_HASH,
        'Notebook 09 outputs use a different protocol.')
require(baseline_manifest['input_csv_sha256'] == input_hashes[INPUT],
        'Notebook 09 outputs use another rental CSV. Investigate the change first.')
require(baseline_manifest['source_workbook_sha256'] == EXPECTED_SOURCE_SHA256,
        'Notebook 09 source vintage changed.')
bq = baseline_manifest['quality']
require(bq['validation_targets_per_model'] == 589 and bq['holdout_forecasts'] == 0
        and bq['holdout_scores'] == 0 and bq['csv_readback_matches'] is True,
        'Notebook 09 is incomplete or the reserved final period has already been scored.')
for key in ['source_sha256','target','property_type','first_quarter','last_quarter',
            'validation_start','validation_end','holdout_start','holdout_end','forecast_horizon_quarters']:
    require(ridge_protocol[key] == protocol[key], f'Experiment/baseline scope differs: {key}')
FEATURES = ['rent_lag_1','rent_lag_2','rent_lag_3','rent_lag_4',
            'target_is_q2','target_is_q3','target_is_q4']
LAGS = [1, 2, 3, 4]
RIDGE_NAME = 'ridge_lags_seasonal_alpha1'
require(ridge_protocol['numeric_features'] == FEATURES and ridge_protocol['lag_quarters'] == LAGS,
        'Feature specification changed; retain this experiment and create a new reviewed protocol instead.')
require(ridge_protocol['estimator'] == {'class':'Ridge','alpha':1.0,'fit_intercept':True,'solver':'svd'},
        'Unreviewed Ridge settings.')
require(ridge_protocol['model_name'] == RIDGE_NAME and ridge_protocol['minimum_training_examples'] == 40,
        'Unreviewed model specification.')
RIDGE_HASH = canonical_hash(ridge_protocol)
require(RIDGE_HASH == 'e98c4793b0e3c4f9bdfc0bfad9e54a9e1507fcc3851e49d8f1c21ec2672475e8',
        'The fixed Ridge protocol changed. Preserve this experiment before defining a new one.')
VALIDATION = pd.period_range(protocol['validation_start'], protocol['validation_end'], freq='Q-DEC')
HOLDOUT = pd.period_range(protocol['holdout_start'], protocol['holdout_end'], freq='Q-DEC')
require(len(VALIDATION) == 19 and len(HOLDOUT) == 8 and VALIDATION[-1] + 1 == HOLDOUT[0],
        'Chronological split changed.')
SERIES_KEYS = ['rental_area_name','property_type']
TARGET_KEYS = SERIES_KEYS + ['target_quarter']
MODEL_COLUMNS = SERIES_KEYS + ['quarter','median_weekly_rent','source_file','source_sheet','source_median_cell']

def development_only(frame):
    periods = pd.PeriodIndex(frame['quarter'], freq='Q-DEC')
    return frame.loc[periods <= VALIDATION[-1], MODEL_COLUMNS].copy()

development = development_only(panel)
require(len(development) == 3038, 'Unexpected development history (31 x 98 quarters).')
LGAS = sorted(development['rental_area_name'].unique())
del panel  # Discard the full table before creating features or fitting a model.

# Read existing baseline results using THEIR local manifest, not portable hashes.
BASE_FILES = ['validation_predictions.csv', 'validation_metrics_overall.csv',
              'validation_metrics_by_lga.csv','validation_metrics_by_quarter.csv']
baseline_tables = {}
for filename in BASE_FILES:
    p = BASE_DIR / filename
    require(p.is_file() and sha256_file(p) == baseline_manifest['output_sha256'][filename],
            f'Baseline output missing/changed: {filename}. Do not bypass the manifest.')
    baseline_tables[filename] = pd.read_csv(p, dtype=baseline_manifest['output_schemas'][filename],
                                          keep_default_na=False)
baseline_predictions = baseline_tables['validation_predictions.csv']
require(len(baseline_predictions) == 1178 and
        not baseline_predictions.duplicated(['model', *TARGET_KEYS]).any(), 'Baseline keys changed.')
require(set(baseline_predictions['model']) == {'last_observation','seasonal_naive'}
        and baseline_predictions['target_quarter'].isin(VALIDATION.astype(str)).all(),
        'Unexpected baseline method or forecast period.')

# Protect earlier notebooks, their evidence and the published app. Only our new outputs may be written.
more_protected = [RIDGE_PROTOCOL_PATH, BASE_MANIFEST_PATH]
more_protected += [p for p in BASE_DIR.rglob('*') if p.is_file()]
more_protected += [p for p in (ROOT/'notebooks').glob('*.ipynb') if not p.name.startswith('10_')]
more_protected += [p for p in (ROOT/'src').rglob('*.py')]
more_protected += [ROOT/'docs/rental_forecast_baselines.md']
for p in more_protected:
    if p.is_file(): input_hashes[p] = sha256_file(p)
OUT = ROOT / 'data/interim/rental_ridge_validation'
MANIFEST_PATH = OUT / 'ridge_manifest.json'
if MANIFEST_PATH.is_file():
    old = read_json(MANIFEST_PATH)
    require(old.get('ridge_protocol_sha256') == RIDGE_HASH and
            old.get('input_csv_sha256') == input_hashes[INPUT],
            'An existing Ridge run uses different inputs/settings. Preserve it for review.')
print('EXPERIMENT INPUT CHECKPOINT PASSED')
print('Baseline forecasts verified:', len(baseline_predictions))
print('Validation: 2019Q1-2023Q3 | Final period reserved: 2023Q4-2025Q3')
print('Ridge: separate LGA models, alpha=1.0, four rent lags + three calendar indicators')
print('Parameter combinations searched: 0 | Transport tables loaded: 0')


EXPERIMENT INPUT CHECKPOINT PASSED
Baseline forecasts verified: 1178
Validation: 2019Q1-2023Q3 | Final period reserved: 2023Q4-2025Q3
Ridge: separate LGA models, alpha=1.0, four rent lags + three calendar indicators
Parameter combinations searched: 0 | Transport tables loaded: 0


## 3. Build historical features and fit a training-only model

A *feature* is an input supplied to the model. A *lag* is an observation from an earlier quarter. The feature builder returns predictors and lineage only—**not target-quarter rent or published count**.

Ridge is a linear regression with a squared-coefficient penalty. With standardized inputs, its fitting objective is:

$$\sum_i (y_i-\hat y_i)^2 + \alpha\sum_j\beta_j^2,\qquad\alpha=1.$$

The penalty discourages large slope coefficients; it does not guarantee better forecasting. The intercept is fitted separately. All seven inputs are standardized from the training examples for each LGA/origin, never from the whole series or validation period.

`fit_one_origin()` removes current/future labels before joining training outcomes. It then predicts the single target row. An independent NumPy solution to the Ridge equations checks the prediction. The fitted mean, scale and coefficients are retained for auditing—not treated as causal effects.

References: [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [leakage and pipelines](https://scikit-learn.org/stable/common_pitfalls.html).


In [3]:
def make_lag_features(frame):
    """Create predictor-only rows. No target-quarter rent or count is returned."""
    require(set(MODEL_COLUMNS).issubset(frame.columns), 'Missing history columns.')
    require(not frame.duplicated(SERIES_KEYS + ['quarter']).any(), 'Duplicate rental key.')
    require(frame[SERIES_KEYS].notna().all().all(), 'Missing series identifiers.')
    pieces = []
    for _, group in frame.groupby(SERIES_KEYS, sort=True, observed=True):
        g = group.copy()
        g['_period'] = pd.PeriodIndex(g['quarter'], freq='Q-DEC')
        g = g.sort_values('_period', kind='stable').reset_index(drop=True)
        idx = pd.PeriodIndex(g['_period'], freq='Q-DEC')
        require(len(g) > 4 and idx.equals(pd.period_range(idx.min(), idx.max(), freq='Q-DEC')),
                'Gap in history or too few quarters for four lags.')
        values = g['median_weekly_rent'].to_numpy(dtype=float)
        require(np.isfinite(values).all() and (values > 0).all(), 'Nonpositive or nonfinite rental input.')
        require(g['source_median_cell'].astype('string').str.fullmatch(r'[A-Z]+[1-9][0-9]*').all(),
                'Missing/malformed source cell.')
        f = g[SERIES_KEYS + ['source_file','source_sheet']].copy()
        f['target_quarter'] = idx.astype(str)
        f['target_index'] = idx.asi8
        f['forecast_origin_quarter'] = (idx-1).astype(str)
        for lag in LAGS:
            f[f'rent_lag_{lag}'] = g['median_weekly_rent'].astype(float).shift(lag)
            f[f'lag_{lag}_quarter'] = g['quarter'].shift(lag)
            f[f'lag_{lag}_source_cell'] = g['source_median_cell'].shift(lag)
        for q in [2,3,4]:
            f[f'target_is_q{q}'] = (idx.quarter == q).astype(float)
        f = f.iloc[4:].copy()  # Exactly four warm-up targets; never drop missing rows silently.
        require(f[FEATURES].notna().all().all(), 'Missing predictor after warm-up.')
        pieces.append(f)
    require(bool(pieces), 'No rental series available.')
    return pd.concat(pieces, ignore_index=True).sort_values(TARGET_KEYS, kind='stable').reset_index(drop=True)


def outcome_table(frame):
    result = frame[SERIES_KEYS+['quarter','median_weekly_rent','source_median_cell']].rename(columns={
        'quarter':'target_quarter','median_weekly_rent':'actual_weekly_rent_aud',
        'source_median_cell':'target_source_median_cell'})
    require(not result.duplicated(TARGET_KEYS).any(), 'Duplicate target outcome.')
    return result.copy()


def fit_one_origin(feature_frame, outcomes, area, property_type, target, minimum_examples=40):
    """Fit a fresh preprocessing/model pipeline using this LGA's earlier targets only."""
    target = pd.Period(target, freq='Q-DEC')
    f = feature_frame.loc[feature_frame.rental_area_name.eq(area)
                          & feature_frame.property_type.eq(property_type)].copy()
    require(not f.duplicated(TARGET_KEYS).any(), 'Duplicate feature key.')
    training = f.loc[f.target_index.lt(target.ordinal)].copy()
    predict_row = f.loc[f.target_index.eq(target.ordinal)].copy()
    require(len(predict_row) == 1, 'Exactly one forecast target is required.')
    require(len(training) >= minimum_examples, 'Insufficient training examples.')
    require(training.target_index.max() == target.ordinal-1, 'Training must end at the preceding quarter.')
    # Target/future outcome rows are removed BEFORE their values are joined or fitted.
    label_period = pd.PeriodIndex(outcomes.target_quarter, freq='Q-DEC')
    labels = outcomes.loc[outcomes.rental_area_name.eq(area) & outcomes.property_type.eq(property_type)
                          & (label_period < target)].copy()
    require(not labels.duplicated(TARGET_KEYS).any(), 'Duplicate training outcome.')
    training = training.merge(labels, on=TARGET_KEYS, how='left', validate='one_to_one', indicator=True)
    require(training['_merge'].eq('both').all(), 'A training outcome is missing.')
    training = training.sort_values('target_index', kind='stable')
    require(training[FEATURES+['actual_weekly_rent_aud']].notna().all().all(), 'Missing training value.')
    X = training[FEATURES].astype(float)
    y = training.actual_weekly_rent_aud.to_numpy(dtype=float)
    X_target = predict_row[FEATURES].astype(float)
    require(np.isfinite(X.to_numpy()).all() and np.isfinite(y).all() and
            np.isfinite(X_target.to_numpy()).all(), 'Nonfinite model input.')
    pipeline = Pipeline([('scale', StandardScaler()),
                         ('ridge', Ridge(alpha=1.0, fit_intercept=True, solver='svd'))])
    pipeline.fit(X, y)
    forecast = float(pipeline.predict(X_target)[0])
    scaler = pipeline.named_steps['scale']; estimator = pipeline.named_steps['ridge']
    require(int(scaler.n_samples_seen_) == len(X), 'Scaler sample count differs from training rows.')
    np.testing.assert_allclose(scaler.mean_, X.to_numpy().mean(axis=0), rtol=1e-12, atol=1e-12)
    np.testing.assert_allclose(scaler.var_, X.to_numpy().var(axis=0), rtol=1e-12, atol=1e-12)

    # Independent closed-form Ridge calculation checks both the fit and prediction.
    z = (X.to_numpy()-scaler.mean_) / scaler.scale_
    z_mean = z.mean(axis=0)
    z_centered = z-z_mean
    y_mean = y.mean()
    coefficient_check = np.linalg.solve(z_centered.T @ z_centered + np.eye(len(FEATURES)),
                                       z_centered.T @ (y-y_mean))
    target_z = (X_target.to_numpy()[0]-scaler.mean_) / scaler.scale_
    independent = float((target_z-z_mean) @ coefficient_check+y_mean)
    np.testing.assert_allclose(forecast, independent, rtol=1e-9, atol=1e-8)
    require(np.isfinite(forecast), 'Nonfinite Ridge forecast.')
    raw_coefs = estimator.coef_ / scaler.scale_
    raw_intercept = float(estimator.intercept_ - raw_coefs @ scaler.mean_)
    np.testing.assert_allclose(forecast, raw_intercept+X_target.to_numpy()[0]@raw_coefs,
                               rtol=1e-10, atol=1e-8)
    base = {**dict(zip(SERIES_KEYS, [area, property_type])), 'target_quarter':str(target),
            'forecast_origin_quarter':str(target-1)}
    prediction = {**base,'model':RIDGE_NAME,'forecast_weekly_rent_aud':forecast}
    audit = {**base, 'training_first_target':str(training.target_quarter.min()),
             'training_last_target':str(training.target_quarter.max()), 'training_rows':len(training),
             'alpha':1.0, 'scaler_training_rows':int(scaler.n_samples_seen_),
             'raw_intercept':raw_intercept, 'independent_forecast_aud_week':independent,
             'training_targets_at_or_after_forecast_target':0}
    coefficients = [{**base,'feature':name,'training_mean':float(scaler.mean_[j]),
                     'training_scale':float(scaler.scale_[j]),'coefficient_standardized':float(estimator.coef_[j]),
                     'coefficient_raw_units':float(raw_coefs[j])} for j,name in enumerate(FEATURES)]
    p = predict_row.iloc[0]
    lineage = []
    for lag in LAGS:
        require(pd.Period(p[f'lag_{lag}_quarter'],freq='Q-DEC') == target-lag,
                'Feature lineage is not the correct historical quarter.')
        lineage.append({**base,'feature':f'rent_lag_{lag}','lag_quarters':lag,
                        'source_quarter':p[f'lag_{lag}_quarter'],'source_value_aud_week':float(p[f'rent_lag_{lag}']),
                        'source_file':p.source_file,'source_sheet':p.source_sheet,
                        'source_median_cell':p[f'lag_{lag}_source_cell']})
    return prediction,audit,coefficients,lineage


def metric_table(frame, groups):
    require(len(frame) > 0 and np.isfinite(frame.error_aud_week).all(), 'Metrics require finite errors.')
    rows=[]
    by=groups[0] if len(groups)==1 else groups
    for key,g in frame.groupby(by, sort=True, observed=True):
        keys=(key,) if len(groups)==1 else key
        e=g.error_aud_week.to_numpy(dtype=float)
        rows.append({**dict(zip(groups,keys)), 'n_forecasts':len(e),
                     'mae_aud_week':float(np.mean(np.abs(e))),
                     'rmse_aud_week':float(np.sqrt(np.mean(e*e))),
                     'mean_signed_error_aud_week':float(e.mean())})
    return pd.DataFrame(rows)

print('Predictor-only feature builder and per-origin Ridge fit defined.')
print('No future/current-quarter targets are passed to model.fit().')


Predictor-only feature builder and per-origin Ridge fit defined.
No future/current-quarter targets are passed to model.fit().


## 4. Test the rules on small fictional examples

These fourteen self-check groups run before the actual fits. They reject duplicates, quarter gaps and missing data, check lag/source references and verify that one LGA's data cannot change another LGA's separately fitted model.

The key leakage test changes all rents at or after a chosen target and requires the already-issued forecast to remain identical. Another test removes current/future outcome rows entirely: the model still must predict from history alone. A separate check confirms the final reserved period is discarded before feature construction.

These are notebook self-checks, **not extra cases in the existing pytest suite**.


In [4]:
# These fixtures contain no official data and do not write files.
def toy_series(name='Fixture A', shift=0.0):
    p = pd.period_range('2005Q1','2011Q4',freq='Q-DEC')
    vals = [200.0 + 3*i + (i%4)*2 + shift for i in range(len(p))]
    return pd.DataFrame({'rental_area_name':name,'property_type':'2 bedroom flats',
                         'quarter':p.astype(str),'median_weekly_rent':vals,
                         'source_file':'fictional.xlsx','source_sheet':'fixture',
                         'source_median_cell':[f'A{i+1}' for i in range(len(p))]})

def expect_error(call, phrase):
    try: call()
    except ValueError as exc:
        require(phrase in str(exc), f'Unexpected rejection: {exc}')
    else: raise AssertionError(f'Expected rejection: {phrase}')

self_checks=[]
toy=toy_series(); toy_f=make_lag_features(toy); toy_y=outcome_table(toy)
first=toy_f.iloc[0]
require(first.target_quarter=='2006Q1' and first.rent_lag_1==215 and first.rent_lag_4==200,
        'Wrong toy lag values.')
require(first.lag_1_source_cell=='A4' and first.lag_4_source_cell=='A1','Wrong toy source cells.')
self_checks.append('known historical values, quarter labels and source cells')
pd.testing.assert_frame_equal(toy_f,make_lag_features(toy.sample(frac=1,random_state=21)))
self_checks.append('row-order independence')
expect_error(lambda:make_lag_features(pd.concat([toy,toy.iloc[[0]]],ignore_index=True)),'Duplicate')
self_checks.append('duplicate-quarter rejection')
expect_error(lambda:make_lag_features(toy.drop(index=9)),'Gap')
self_checks.append('missing-quarter rejection')
bad=toy.copy();bad.loc[4,'median_weekly_rent']=np.nan
expect_error(lambda:make_lag_features(bad),'Nonpositive or nonfinite')
self_checks.append('missing-value rejection without filling')
combined_toy=pd.concat([toy,toy_series('Fixture B',700)],ignore_index=True)
combined_f=make_lag_features(combined_toy)
require(combined_f.loc[combined_f.rental_area_name.eq('Fixture B')].iloc[0].rent_lag_4==900,
        'Features mixed LGA histories.')
self_checks.append('lags remain inside each LGA')
for q in [1,2,3,4]:
    r=toy_f.loc[pd.PeriodIndex(toy_f.target_quarter,freq='Q-DEC').quarter==q].iloc[0]
    require([r[f'target_is_q{k}'] for k in [2,3,4]]==[float(q==k) for k in [2,3,4]],'Bad calendar encoding.')
self_checks.append('target quarter is known calendar information')
T='2010Q1'
result=fit_one_origin(toy_f,toy_y,'Fixture A','2 bedroom flats',T,minimum_examples=8)
require(result[1]['training_last_target']=='2009Q4','Bad forecast origin.')
self_checks.append('training origin and independent Ridge solution')
require(result[1]['training_rows']==16 and result[1]['scaler_training_rows']==16,'Scaler saw target rows.')
self_checks.append('preprocessing fitted only on training examples')

# Mutating current/future actual rents must not change the forecast issued at T.
mut=toy.copy();future=pd.PeriodIndex(mut.quarter,freq='Q-DEC')>=pd.Period(T,freq='Q-DEC')
mut.loc[future,'median_weekly_rent']+=100000
mut_f=make_lag_features(mut); mut_y=outcome_table(mut)
after=fit_one_origin(mut_f,mut_y,'Fixture A','2 bedroom flats',T,minimum_examples=8)
np.testing.assert_allclose(result[0]['forecast_weekly_rent_aud'],after[0]['forecast_weekly_rent_aud'],rtol=0,atol=1e-10)
self_checks.append('current and future target mutation cannot change an issued forecast')

# Completely remove current/future outcomes; feature values remain available from past history.
past_y=toy_y.loc[pd.PeriodIndex(toy_y.target_quarter,freq='Q-DEC')<pd.Period(T,freq='Q-DEC')]
without=fit_one_origin(toy_f,past_y,'Fixture A','2 bedroom flats',T,minimum_examples=8)
np.testing.assert_allclose(result[0]['forecast_weekly_rent_aud'],without[0]['forecast_weekly_rent_aud'],atol=1e-10)
self_checks.append('prediction succeeds without current/future outcomes')

isolated=fit_one_origin(combined_f,outcome_table(combined_toy),'Fixture A','2 bedroom flats',T,minimum_examples=8)
np.testing.assert_allclose(result[0]['forecast_weekly_rent_aud'],isolated[0]['forecast_weekly_rent_aud'],atol=1e-10)
self_checks.append('other LGA outcomes cannot affect the separate model')
expect_error(lambda:fit_one_origin(toy_f,toy_y,'Fixture A','2 bedroom flats','2006Q1',minimum_examples=8),
             'Insufficient')
self_checks.append('insufficient-training rejection')
# Final-period values are discarded by the same development-only selector used above.
toy_full=pd.concat([toy, toy.assign(quarter=pd.period_range('2024Q1','2030Q4',freq='Q-DEC').astype(str))],ignore_index=True)
a=development_only(toy_full)
toy_full.loc[pd.PeriodIndex(toy_full.quarter,freq='Q-DEC')>=HOLDOUT[0],'median_weekly_rent']=999999
pd.testing.assert_frame_equal(a,development_only(toy_full))
self_checks.append('reserved/future outcomes excluded before feature construction')
print(f'{len(self_checks)} Ridge self-check groups passed.')
print('These are notebook checks, not additions to the pytest suite.')


14 Ridge self-check groups passed.
These are notebook checks, not additions to the pytest suite.


## 5. Independently check the real features and previous baselines

There are **31 × (98 − 4) = 2,914** development feature rows. Each has four historical rent inputs, so **11,656 value/quarter/cell checks** are performed using exact source keys rather than the shift operation that built them.

The 1,178 baseline predictions are also rechecked against the unchanged history. This makes the comparison use the actual saved notebook-09 evidence; it does not trust a rounded score copied from the chat.

The preview is for Melbourne at the first two validation targets. Its feature cells refer only to earlier quarters.


In [5]:
features = make_lag_features(development)
require(len(features)==2914, 'Expected 31 x 94 supervised predictor rows (four warm-up quarters removed).')
require(features.target_quarter.min()=='2000Q2' and features.target_quarter.max()=='2023Q3',
        'Unexpected supervised feature period.')
require('median_weekly_rent' not in features and 'published_count' not in features,
        'Target or count leaked into predictors.')
require(pd.PeriodIndex(features.target_quarter,freq='Q-DEC').max() < HOLDOUT[0],
        'Final-period feature rows must not be constructed.')

# Check every lag via an independent exact-key lookup, not shift() or model.fit().
truth=development.set_index(SERIES_KEYS+['quarter'])
lag_entries_checked=0
for row in features.itertuples(index=False):
    target=pd.Period(row.target_quarter,freq='Q-DEC')
    for lag in LAGS:
        original=truth.loc[(row.rental_area_name,row.property_type,str(target-lag))]
        require(float(getattr(row,f'rent_lag_{lag}'))==float(original.median_weekly_rent),
                'Lag value differs from independent source lookup.')
        require(getattr(row,f'lag_{lag}_quarter')==str(target-lag)
                and getattr(row,f'lag_{lag}_source_cell')==original.source_median_cell,
                'Lag quarter/cell differs from source lookup.')
        lag_entries_checked+=1
require(lag_entries_checked==11656,'Lag-check count changed.')

# Recheck the saved baseline predictions and actuals against the unchanged history.
for row in baseline_predictions.itertuples(index=False):
    target=pd.Period(row.target_quarter,freq='Q-DEC')
    lag=1 if row.model=='last_observation' else 4
    expected=truth.loc[(row.rental_area_name,row.property_type,str(target-lag))]
    actual=truth.loc[(row.rental_area_name,row.property_type,str(target))]
    require(float(row.forecast_weekly_rent_aud)==float(expected.median_weekly_rent)
            and row.forecast_source_quarter==str(target-lag)
            and row.forecast_source_median_cell==expected.source_median_cell,
            'Saved baseline forecast differs from its historical source.')
    require(float(row.actual_weekly_rent_aud)==float(actual.median_weekly_rent)
            and row.target_source_median_cell==actual.source_median_cell,
            'Saved baseline actual differs from the input rental table.')

print('FEATURE CHECKPOINT PASSED')
print('Supervised predictor rows: 2914 | Columns used by Ridge: 7')
print('Independent historical value/quarter/cell matches:',lag_entries_checked)
print('Previous baseline prediction rows verified: 1178')
display(features.loc[features.rental_area_name.eq('Melbourne') &
                     features.target_quarter.isin(['2019Q1','2019Q2']),
                    SERIES_KEYS+['target_quarter','forecast_origin_quarter',*FEATURES]])


FEATURE CHECKPOINT PASSED
Supervised predictor rows: 2914 | Columns used by Ridge: 7
Independent historical value/quarter/cell matches: 11656
Previous baseline prediction rows verified: 1178


,rental_area_name,property_type,target_quarter,forecast_origin_quarter,rent_lag_1,rent_lag_2,rent_lag_3,rent_lag_4,target_is_q2,target_is_q3,target_is_q4
1673,Melbourne,2 bedroom flats,2019Q1,2018Q4,580.0,593.0,580.0,580.0,0.0,0.0,0.0
1674,Melbourne,2 bedroom flats,2019Q2,2019Q1,600.0,580.0,593.0,580.0,1.0,0.0,0.0


## 6. Make 589 rolling Ridge forecasts

For each LGA and each of the 19 validation targets, fit a fresh model using earlier supervised observations. After one target has occurred, its published outcome may enter the next origin's training data. This is repeated one-step evaluation, **not a 19-quarter forecast issued at one date**.

Progress prints one line after each LGA's 19 fits. Do not repeatedly press Run while the cell is executing. We preserve all 589 fits; no LGA is dropped because its predictions look poor. No fitted model is saved into the deployed app.

References: [rolling-origin evaluation](https://otexts.com/fpp3/tscv.html).


In [6]:
# 19 expanding-window fits per LGA. No alpha search and no holdout fits.
outcomes=outcome_table(development)
rows=[]; audit_rows=[]; coefficient_rows=[]; lineage_rows=[]
for i,area in enumerate(LGAS,start=1):
    area_features=features.loc[features.rental_area_name.eq(area)].copy()
    area_outcomes=outcomes.loc[outcomes.rental_area_name.eq(area)].copy()
    for target in VALIDATION:
        pred, audit, coefs, lineage=fit_one_origin(
            area_features, area_outcomes, area, protocol['property_type'], target,
            minimum_examples=ridge_protocol['minimum_training_examples'])
        rows.append(pred); audit_rows.append(audit)
        coefficient_rows.extend(coefs); lineage_rows.extend(lineage)
    print(f'{i:02d}/31 {area}: 19 validation forecasts completed.',flush=True)
ridge_predictions=pd.DataFrame(rows).sort_values(TARGET_KEYS,kind='stable').reset_index(drop=True)
fit_audit=pd.DataFrame(audit_rows).sort_values(TARGET_KEYS,kind='stable').reset_index(drop=True)
coefficient_audit=pd.DataFrame(coefficient_rows).sort_values([*TARGET_KEYS,'feature'],kind='stable').reset_index(drop=True)
feature_lineage=pd.DataFrame(lineage_rows).sort_values([*TARGET_KEYS,'lag_quarters'],kind='stable').reset_index(drop=True)
require(len(ridge_predictions)==589 and not ridge_predictions.duplicated(TARGET_KEYS).any(),
        'Unexpected Ridge forecast keys/count.')
require(ridge_predictions.target_quarter.isin(VALIDATION.astype(str)).all(),'Ridge targets outside validation.')
require(len(fit_audit)==589 and len(coefficient_audit)==4123 and len(feature_lineage)==2356,
        'Fit evidence counts changed.')
require(fit_audit.training_rows.min()==75 and fit_audit.training_rows.max()==93,
        'Unexpected expanding-training sizes.')
require((pd.PeriodIndex(fit_audit.training_last_target,freq='Q-DEC') ==
         pd.PeriodIndex(fit_audit.target_quarter,freq='Q-DEC')-1).all(), 'Model trained after its origin.')
print('RIDGE FORECAST CHECKPOINT PASSED')
print('Fitted LGA/origin models: 589 | Training examples per fit: 75-93')
print('Independent closed-form predictions matched: 589')
print('Forecast lag-source records: 2356 | Final-period forecasts: 0')


01/31 Banyule: 19 validation forecasts completed.
02/31 Bayside: 19 validation forecasts completed.
03/31 Boroondara: 19 validation forecasts completed.
04/31 Brimbank: 19 validation forecasts completed.
05/31 Cardinia: 19 validation forecasts completed.
06/31 Casey: 19 validation forecasts completed.
07/31 Darebin: 19 validation forecasts completed.
08/31 Frankston: 19 validation forecasts completed.
09/31 Glen Eira: 19 validation forecasts completed.
10/31 Greater Dandenong: 19 validation forecasts completed.
11/31 Hobsons Bay: 19 validation forecasts completed.
12/31 Hume: 19 validation forecasts completed.
13/31 Kingston: 19 validation forecasts completed.
14/31 Knox: 19 validation forecasts completed.
15/31 Manningham: 19 validation forecasts completed.
16/31 Maribyrnong: 19 validation forecasts completed.
17/31 Maroondah: 19 validation forecasts completed.
18/31 Melbourne: 19 validation forecasts completed.
19/31 Melton: 19 validation forecasts completed.
20/31 Merri-bek: 19 vali

## 7. Score the same targets for all three methods

The predictions already exist before the actual target values are attached. The resulting comparison has **589 × 3 = 1,767 rows**.

- **MAE:** average absolute forecast error, in AUD/week. Primary comparison; lower is better.
- **RMSE:** square root of the average squared error. Larger misses receive more weight.
- **Mean signed error:** average of `forecast − actual`; negative means underprediction on average. Cancellation can hide large errors.

Pooled errors use equal weight per LGA-quarter. They are not weighted by the source's published count. Pooled RMSE is checked against the square root of the weighted squared per-LGA RMSEs—not the average RMSE.

The printed percentage is `(Ridge MAE / last-observation MAE − 1) × 100`; a positive percentage means Ridge is worse. This is a descriptive validation comparison, not statistical significance or final-test evidence.


In [7]:
# Predictions have already been made. Only now attach each current outcome for scoring.
CORE = ['model',*TARGET_KEYS,'forecast_origin_quarter','forecast_weekly_rent_aud']
predictions=pd.concat([baseline_predictions[CORE],ridge_predictions[CORE]],ignore_index=True)
require(not predictions.duplicated(['model',*TARGET_KEYS]).any(),'Duplicate prediction key.')
comparison=predictions.merge(outcomes,on=TARGET_KEYS,how='left',validate='many_to_one',indicator=True)
require(comparison['_merge'].eq('both').all(),'Missing scored outcome.')
comparison=comparison.drop(columns='_merge').sort_values(['model',*TARGET_KEYS],kind='stable').reset_index(drop=True)
comparison['error_aud_week']=comparison.forecast_weekly_rent_aud-comparison.actual_weekly_rent_aud
comparison['absolute_error_aud_week']=comparison.error_aud_week.abs()
comparison['squared_error']=comparison.error_aud_week.pow(2)
require(len(comparison)==1767 and comparison.groupby('model').size().eq(589).all(),'Unequal validation coverage.')
require(not pd.PeriodIndex(comparison.target_quarter,freq='Q-DEC').isin(HOLDOUT).any(),
        'Final evaluation was accidentally scored.')
ridge_scored=comparison.loc[comparison.model.eq(RIDGE_NAME)].reset_index(drop=True)
overall_metrics=metric_table(comparison,['model'])
lga_metrics=metric_table(comparison,['model',*SERIES_KEYS])
quarter_metrics=metric_table(comparison,['model','target_quarter'])
METRIC_COLUMNS=['n_forecasts','mae_aud_week','rmse_aud_week','mean_signed_error_aud_week']
for model in ['last_observation','seasonal_naive']:
    before=baseline_tables['validation_metrics_overall.csv'].set_index('model').loc[model,METRIC_COLUMNS]
    after=overall_metrics.set_index('model').loc[model,METRIC_COLUMNS]
    np.testing.assert_allclose(before.to_numpy(dtype=float),after.to_numpy(dtype=float),rtol=1e-12,atol=1e-12)
require(len(lga_metrics)==93 and lga_metrics.n_forecasts.eq(19).all(),'Per-LGA scoring count changed.')
require(len(quarter_metrics)==57 and quarter_metrics.n_forecasts.eq(31).all(),'Per-quarter scoring count changed.')
for m in overall_metrics.model:
    pooled=overall_metrics.loc[overall_metrics.model.eq(m)].iloc[0]
    g=lga_metrics.loc[lga_metrics.model.eq(m)]
    np.testing.assert_allclose(pooled.mae_aud_week,g.mae_aud_week.mean(),rtol=1e-12,atol=1e-12)
    np.testing.assert_allclose(pooled.rmse_aud_week,np.sqrt(np.mean(g.rmse_aud_week**2)),rtol=1e-12,atol=1e-12)
paired=lga_metrics.pivot(index=SERIES_KEYS,columns='model',values='mae_aud_week').reset_index()
paired.columns.name=None
paired=paired.rename(columns={'last_observation':'last_observation_mae',RIDGE_NAME:'ridge_mae',
                              'seasonal_naive':'seasonal_naive_mae'})
paired['ridge_minus_last_mae_aud_week']=paired.ridge_mae-paired.last_observation_mae
paired['ridge_vs_last']=np.where(np.isclose(paired.ridge_minus_last_mae_aud_week,0,atol=1e-10),'tie',
                                 np.where(paired.ridge_minus_last_mae_aud_week<0,'ridge_lower_mae','last_lower_mae'))
last_mae=float(overall_metrics.set_index('model').loc['last_observation','mae_aud_week'])
ridge_mae=float(overall_metrics.set_index('model').loc[RIDGE_NAME,'mae_aud_week'])
mae_change_pct=100.0*(ridge_mae/last_mae-1.0)
MODEL_LABELS={'last_observation':'Last observation','seasonal_naive':'Seasonal naive',
              RIDGE_NAME:'Ridge (fixed alpha=1)'}
MODEL_ORDER=['last_observation','seasonal_naive',RIDGE_NAME]
print('VALIDATION COMPARISON CHECKPOINT PASSED')
display(overall_metrics.round(4))
print(f'Ridge MAE change versus last observation: {mae_change_pct:+.2f}% (negative is better).')
print('Per-LGA descriptive comparison:')
display(paired.ridge_vs_last.value_counts().rename_axis('comparison').reset_index(name='lga_count'))
print('These are validation findings, not holdout evidence, significance tests or a deployment decision.')


VALIDATION COMPARISON CHECKPOINT PASSED


,model,n_forecasts,mae_aud_week,rmse_aud_week,mean_signed_error_aud_week
0,last_observation,589,10.9728,16.2303,-4.3582
1,ridge_lags_seasonal_alpha1,589,13.2135,18.6421,-1.9758
2,seasonal_naive,589,25.2937,38.9464,-13.0934


Ridge MAE change versus last observation: +20.42% (negative is better).
Per-LGA descriptive comparison:


,comparison,lga_count
0,last_lower_mae,26
1,ridge_lower_mae,5


These are validation findings, not holdout evidence, significance tests or a deployment decision.


## 8. Charts 1–2: overall error and changes across validation quarters

The first chart compares the three MAEs on identical targets. The second shows how average error varies over the 19 quarters. Neither graph includes the final reserved period.

Lower error for the trained method is a hypothesis to evaluate, not an assumption. Do not modify the protocol after looking at the charts to hide an unfavourable result.


In [8]:
figures={}
plot_metrics=overall_metrics.assign(method=overall_metrics.model.map(MODEL_LABELS))
figures['01_validation_mae.html']=px.bar(
    plot_metrics,x='method',y='mae_aud_week',
    category_orders={'method':[MODEL_LABELS[m] for m in MODEL_ORDER]},
    hover_data={'n_forecasts':True,'rmse_aud_week':':.2f','mean_signed_error_aud_week':':.2f'},
    labels={'method':'Method','mae_aud_week':'Validation MAE (AUD/week)'},
    title='Fixed Ridge versus two baselines | 2019Q1-2023Q3')
figures['01_validation_mae.html'].update_layout(height=440)
figures['01_validation_mae.html'].show()
qplot=quarter_metrics.assign(method=quarter_metrics.model.map(MODEL_LABELS)).sort_values(['model','target_quarter'])
figures['02_validation_mae_by_quarter.html']=px.line(
    qplot,x='target_quarter',y='mae_aud_week',color='method',markers=True,
    category_orders={'method':[MODEL_LABELS[m] for m in MODEL_ORDER]},
    labels={'target_quarter':'Target quarter (validation only)','mae_aud_week':'MAE across 31 LGAs (AUD/week)','method':'Method'},
    title='Error by quarter | repeated one-step forecasts, not one long forecast')
figures['02_validation_mae_by_quarter.html'].update_layout(height=550,margin=dict(t=75,b=110),legend=dict(orientation='h',y=-0.22,x=0,yanchor='top'))
figures['02_validation_mae_by_quarter.html'].show()


## 9. Chart 3: where Ridge helps or hurts relative to last observation

Each bar is **Ridge MAE minus last-observation MAE** for one LGA, using the same 19 targets.

- Left of zero: smaller Ridge error in this validation period.
- Right of zero: smaller last-observation error.

Areas remain in alphabetical order. The figure is not a best-area ranking. A per-LGA improvement can coexist with a worse overall result, and neighbouring/time-dependent errors are not independent statistical replicates.


In [9]:
figures['03_ridge_minus_last_by_lga.html']=px.bar(
    paired,x='ridge_minus_last_mae_aud_week',y='rental_area_name',orientation='h',
    category_orders={'rental_area_name':LGAS},
    hover_data={'ridge_mae':':.2f','last_observation_mae':':.2f','ridge_vs_last':True},
    labels={'rental_area_name':'Rental-source LGA','ridge_minus_last_mae_aud_week':'Ridge MAE minus last-observation MAE (AUD/week)'},
    title='Per-LGA difference | left of zero favours Ridge; right favours last observation')
figures['03_ridge_minus_last_by_lga.html'].add_vline(x=0,line_dash='dash')
figures['03_ridge_minus_last_by_lga.html'].update_layout(height=1000,margin=dict(l=175))
figures['03_ridge_minus_last_by_lga.html'].show()
print('No significance test is claimed. A smaller error in a few areas is not proof of a reliable advantage.')


No significance test is claimed. A smaller error in a few areas is not proof of a reliable advantage.


## 10. Chart 4: explain one LGA's forecasts

Melbourne remains the illustrative area used in previous notebooks; it is not selected because its score is favourable. The chart compares the actual median with three rolling forecasts on the validation targets.

Hover information includes the forecast origin. The outcome's workbook cell is evidence used for scoring, not a model input. To inspect the four historical inputs supplied to Ridge, use `forecast_feature_lineage.csv` after export.


In [10]:
EXAMPLE_LGA='Melbourne'  # Same illustrative LGA as notebook 09, not selected from the scores.
example=comparison.loc[comparison.rental_area_name.eq(EXAMPLE_LGA)].sort_values(['model','target_quarter'])
actual=example[TARGET_KEYS+['actual_weekly_rent_aud']].drop_duplicates().sort_values('target_quarter')
require(len(actual)==19 and len(example)==57,'Incorrect example coverage.')
trace=go.Figure()
trace.add_trace(go.Scatter(x=actual.target_quarter,y=actual.actual_weekly_rent_aud,
                           mode='lines+markers',name='Published actual'))
for model in MODEL_ORDER:
    g=example.loc[example.model.eq(model)]
    trace.add_trace(go.Scatter(x=g.target_quarter,y=g.forecast_weekly_rent_aud,mode='lines+markers',
        name=MODEL_LABELS[model],customdata=g[['forecast_origin_quarter','target_source_median_cell']].to_numpy(),
        hovertemplate='Target: %{x}<br>Forecast: $%{y:.2f}/week<br>Origin: %{customdata[0]}<br>'
                      'Outcome source cell (scoring only): %{customdata[1]}<extra>%{fullData.name}</extra>'))
trace.update_layout(title='Melbourne | validation-only actuals and one-quarter forecasts',
                    xaxis_title='Target quarter',yaxis_title='Nominal median weekly rent (AUD/week)',
                    height=580,margin=dict(t=75,b=110),legend=dict(orientation='h',y=-0.22,x=0,yanchor='top'))
figures['04_melbourne_validation_forecasts.html']=trace
trace.show()
print('The line connects repeated one-step predictions. Later predictions use newly observed earlier rents.')
print('No graph includes the reserved final period or a current-rent estimate.')


The line connects repeated one-step predictions. Later predictions use newly observed earlier rents.
No graph includes the reserved final period or a current-rent estimate.


## 11. Export the research evidence and reload every table

Outputs go only to `data/interim/rental_ridge_validation/`. Nine CSV tables, four self-contained HTML charts and a protocol copy are saved. A manifest and generated findings report follow in the final cell.

`fit_audit.csv` records each training window. `preprocessing_and_coefficients.csv` records training-only means, scales and coefficients for all seven inputs. `forecast_feature_lineage.csv` records the four historical rent values and cells used by each Ridge forecast.

The prior baseline exports, source data, registry, source modules, notebooks 01–09, application and frozen release are protected. Rerunning replaces **only this experiment's** outputs when input/protocol checks agree. Publishing several output files is not a database transaction; do not run simultaneous notebook writers.


In [11]:
exports={
    'ridge_validation_predictions.csv':ridge_scored,
    'validation_predictions_all_methods.csv':comparison,
    'validation_metrics_overall.csv':overall_metrics,
    'validation_metrics_by_lga.csv':lga_metrics,
    'validation_metrics_by_quarter.csv':quarter_metrics,
    'ridge_vs_last_by_lga.csv':paired,
    'fit_audit.csv':fit_audit,
    'preprocessing_and_coefficients.csv':coefficient_audit,
    'forecast_feature_lineage.csv':feature_lineage,
}

def schema_for(frame):
    result={}
    for c,d in frame.dtypes.items():
        if pd.api.types.is_integer_dtype(d): result[c]='int64'
        elif pd.api.types.is_float_dtype(d): result[c]='float64'
        else: result[c]='string'
    return result


def compare_roundtrip(expected,actual):
    require(expected.columns.tolist()==actual.columns.tolist() and len(expected)==len(actual),'CSV schema/row mismatch.')
    for col,kind in schema_for(expected).items():
        if kind=='float64':
            np.testing.assert_allclose(expected[col].to_numpy(dtype=float),actual[col].to_numpy(dtype=float),
                                        rtol=1e-11,atol=1e-10,equal_nan=False)
        elif kind=='int64':
            require(np.array_equal(expected[col].to_numpy(),actual[col].to_numpy()),f'Integer read-back mismatch: {col}')
        else:
            require(expected[col].astype('string').reset_index(drop=True).equals(actual[col].astype('string').reset_index(drop=True)),
                    f'Text read-back mismatch: {col}')

for path,digest in input_hashes.items():
    require(sha256_file(path)==digest,f'Protected input changed: {path.relative_to(ROOT)}')
require(comparison.target_quarter.isin(VALIDATION.astype(str)).all(),'Refusing to export final-period scores.')
OUT.mkdir(parents=True,exist_ok=True)
output_schemas={}
with TemporaryDirectory(prefix='.ridge-stage-',dir=OUT) as temporary:
    stage=Path(temporary)
    for filename,frame in exports.items():
        frame.to_csv(stage/filename,index=False,encoding='utf-8',lineterminator='\n')
        schema=schema_for(frame);output_schemas[filename]=schema
        loaded=pd.read_csv(stage/filename,dtype=schema,keep_default_na=False)
        compare_roundtrip(frame,loaded)
    for filename in exports:
        (stage/filename).replace(OUT/filename)
figure_dir=OUT/'figures';figure_dir.mkdir(exist_ok=True)
for filename,figure in figures.items():
    figure.write_html(figure_dir/filename,include_plotlyjs=True,full_html=True,
                      config={'responsive':True,'displaylogo':False})
(OUT/'ridge_protocol.json').write_text(json.dumps(ridge_protocol,indent=2)+'\n',encoding='utf-8')
print('Nine CSV tables reloaded and matched; four interactive charts saved.')
print('No trained model pickle, final-period forecast or deployment data has been created.')


Nine CSV tables reloaded and matched; four interactive charts saved.
No trained model pickle, final-period forecast or deployment data has been created.


## 12. Record the outcome without changing the deployed release

The report records the actual scores, whether Ridge improved the primary metric, and the limitations of the experiment. It does not declare a production winner merely because fitting and checks succeeded.

We have one workbook vintage, not historical publication/revision snapshots. Recent final-period rents were visible in earlier descriptive analysis; this period is reserved from modelling evaluation, not a pristine blind test. The 2026 transport snapshot remains excluded. No current-rent forecast, final-period score, uncertainty interval, individual-property forecast or recommendation is produced.

This is the end of this fixed candidate experiment. Keep it as evidence before considering any additional model or tuning.


In [12]:
REPORT_PATH=ROOT/'docs/rental_ridge_validation.md'
created_at=datetime.now(timezone.utc).isoformat()
lines=['# HomeReach Melbourne — Fixed Ridge rental validation','',f'Generated (UTC): {created_at}','',
'## Question and fixed design','',
'Compare a trained Ridge model with the two saved baselines for retrospective one-quarter-ahead published median weekly rent.',
'One separate model per LGA/property series; no cross-LGA pooling. Four rental lags plus Q2/Q3/Q4 calendar indicators (Q1 reference).',
'All seven predictors are standardized using training examples only. Ridge alpha=1.0, intercept enabled, SVD solver.',
'One fixed specification was declared before this experiment was scored. No parameter search, transformation, forecast clipping or post-score retuning.',
'Initial history ends 2018Q4. Four lag warm-up quarters are omitted, so supervised training starts 2000Q2.',
'For each validation target, training targets end at the preceding quarter. Models and scalers are refitted at every origin.',
'Rolling validation: 2019Q1–2023Q3. Reserved final evaluation: 2023Q4–2025Q3, with no forecasts or scores produced here.','',
'## Validation results','',
'| Method | Targets | MAE (AUD/week) | RMSE (AUD/week) | Mean signed error (forecast minus actual) |',
'|---|---:|---:|---:|---:|']
for m in MODEL_ORDER:
    r=overall_metrics.set_index('model').loc[m]
    lines.append(f'| {MODEL_LABELS[m]} | {int(r.n_forecasts)} | {r.mae_aud_week:.4f} | {r.rmse_aud_week:.4f} | {r.mean_signed_error_aud_week:.4f} |')
counts=paired.ridge_vs_last.value_counts()
lines += ['',f'Ridge MAE change versus last observation: {mae_change_pct:+.2f}% (negative is better).',
          f'LGAs with lower Ridge MAE: {int(counts.get("ridge_lower_mae",0))}; lower last-observation MAE: {int(counts.get("last_lower_mae",0))}; ties: {int(counts.get("tie",0))}.',
          'All LGA-quarter targets receive equal weight. MAE is primary; RMSE is calculated from all squared errors, not averaged area RMSEs.',
          'A smaller mean signed error does not imply lower absolute error. Positive and negative errors can cancel.',
          ('This fixed Ridge candidate did NOT improve on the last-observation baseline on pooled validation MAE.' if ridge_mae>=last_mae else
           'This fixed Ridge candidate improved pooled validation MAE; this is not final-period or prospective evidence.'),
          'No model-selection winner for deployment, confidence interval or statistical significance claim is made.','',
          '## Leakage and reproducibility checks','',
          '- 589 separate LGA/origin model fits; 75–93 earlier supervised examples per fit.',
          f'- {lag_entries_checked:,} development lag values, quarters and source cells independently checked.',
          '- 589 predictions checked against an independent closed-form Ridge calculation.',
          '- 2,356 forecast lag-source records and 4,123 training-scaler/coefficient records exported.',
          '- Every scaler was checked against its actual training mean, variance and sample count.',
          f'- {len(self_checks)} synthetic self-check groups passed, including current/future-outcome mutation and prediction without target outcomes.',
          '- The 1,178 baseline predictions were verified against their local manifest and original historical values; baseline metrics matched.',
          '- Same 589 validation targets per method; 1,767 combined scored prediction rows.',
          '- Nine CSV tables reloaded and matched; four validation-only charts saved.',
          f'- Negative Ridge predictions: {int((ridge_scored.forecast_weekly_rent_aud<0).sum())}; no clipping was applied.',
          '- No raw transport data, forecasting API, test-period score, current-rent forecast or model deployment.','',
          '## Interpretation limits','',
          'This is a period-based retrospective experiment with one 2025Q3 workbook vintage. Publication delays and historical revisions are not reconstructed.',
          'Recent final-period outcomes appeared in earlier descriptive EDA and the deployed explorer. They are reserved from modelling evaluation now, not a pristine blind test.',
          'The complete rental CSV is checked structurally, but final-period values are removed before predictor construction, fitting, metrics and plots.',
          'Current target-quarter rent is used only for scoring; current or lagged published counts and the 2026 transport snapshot are not predictors.',
          'The original baseline protocol retains its same-series rule; this extension also trains separately within each LGA.',
          'Historic publisher series are not independently harmonised for boundary revisions. Medians concern changing lettings, not fixed-property histories or current listings.',
          'Success of the software checks means the experiment executed as specified; it does not mean the trained model beat the benchmark.',
          'Repeated experiments can overfit the validation set. Record this candidate, including an unfavourable result, before proposing a new specification.','',
          '## Evidence and reproduction','',
          f'Input CSV SHA-256: `{input_hashes[INPUT]}`.',
          f'Baseline protocol SHA-256: `{BASE_PROTOCOL_HASH}`.',
          f'Ridge protocol SHA-256: `{RIDGE_HASH}`.',
          f'Runtime: Python {sys.version.split()[0]}; pandas {pd.__version__}; scikit-learn {sklearn.__version__}; NumPy {np.__version__}.',
          'Run notebooks/10_rental_ridge_validation.ipynb on feat/rental-forecast-baselines after notebook 09 has passed.',
          'Research outputs: data/interim/rental_ridge_validation/. This generated report is replaced on rerun; keep personal notes separately.',
          'No prior notebooks, baseline exports, source data, registry, source modules or deployed release files are overwritten.','',
          '## References','',
          '- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html',
          '- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html',
          '- https://scikit-learn.org/stable/common_pitfalls.html',
          '- https://otexts.com/fpp3/tscv.html','']
REPORT_PATH.parent.mkdir(parents=True,exist_ok=True)
REPORT_PATH.write_text('\n'.join(lines),encoding='utf-8')
for path,digest in input_hashes.items():
    require(sha256_file(path)==digest,f'Protected file changed: {path.relative_to(ROOT)}')
manifest={'created_at_utc':created_at,'builder':'notebooks/10_rental_ridge_validation.ipynb',
          'research_branch':BRANCH,'source_workbook_sha256':EXPECTED_SOURCE_SHA256,
          'input_csv_sha256':input_hashes[INPUT],'baseline_protocol_sha256':BASE_PROTOCOL_HASH,
          'baseline_manifest_sha256':input_hashes[BASE_MANIFEST_PATH],
          'ridge_protocol_sha256':RIDGE_HASH,'ridge_protocol':ridge_protocol,
          'versions':{'python':sys.version.split()[0],'pandas':pd.__version__,'numpy':np.__version__,
                      'sklearn':sklearn.__version__,'plotly':plotly.__version__},
          'quality':{'ridge_forecasts':len(ridge_scored),'combined_forecasts':len(comparison),
                     'validation_targets_per_model':589,'fitted_models':len(fit_audit),
                     'independent_lag_matches':lag_entries_checked,'independent_ridge_prediction_checks':589,
                     'forecast_lineage_rows':len(feature_lineage),'preprocessing_audit_rows':len(coefficient_audit),
                     'baseline_predictions_verified':1178,'baseline_metrics_match':True,
                     'self_check_groups':len(self_checks),'final_period_forecasts':0,'final_period_scores':0,
                     'current_rent_forecast_produced':False,'transport_features_used':False,
                     'parameter_combinations_searched':0,'csv_readback_matches':True,
                     'csv_tables':len(exports),'charts':len(figures),'protected_files_unchanged':True},
          'self_checks':self_checks,'validation_metrics':overall_metrics.to_dict(orient='records'),
          'output_schemas':output_schemas,
          'output_sha256':{str(p.relative_to(OUT)).replace('\\','/'):sha256_file(p)
                           for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='ridge_manifest.json'},
          'report_sha256':sha256_file(REPORT_PATH)}
MANIFEST_PATH.write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
print('RIDGE VALIDATION CHECKPOINT PASSED')
print('Rental LGAs: 31 | Validation quarters: 19 (2019Q1-2023Q3)')
print('Ridge forecasts: 589 | Combined forecasts for 3 methods: 1767')
print('Fitted LGA/origin models: 589 | Earlier training examples per fit: 75-93')
print('Independent historical lag matches: 11656')
print('Independent closed-form Ridge prediction checks: 589')
print('Previous baseline forecasts verified: 1178 | Baseline metrics: matched')
print('Final-holdout forecasts scored: 0')
print('Ridge self-check groups:',len(self_checks),'passed')
print('CSV tables saved and reloaded:',len(exports))
print('Interactive charts saved:',len(figures))
print('Transport features used: False | Current-rent forecast produced: False')
print('Protected inputs, baseline evidence and deployed release unchanged: True')
print('Saved report:',REPORT_PATH.relative_to(ROOT))
print('Saved outputs:',OUT.relative_to(ROOT))
display(overall_metrics.round(4))


RIDGE VALIDATION CHECKPOINT PASSED
Rental LGAs: 31 | Validation quarters: 19 (2019Q1-2023Q3)
Ridge forecasts: 589 | Combined forecasts for 3 methods: 1767
Fitted LGA/origin models: 589 | Earlier training examples per fit: 75-93
Independent historical lag matches: 11656
Independent closed-form Ridge prediction checks: 589
Previous baseline forecasts verified: 1178 | Baseline metrics: matched
Final-holdout forecasts scored: 0
Ridge self-check groups: 14 passed
CSV tables saved and reloaded: 9
Interactive charts saved: 4
Transport features used: False | Current-rent forecast produced: False
Protected inputs, baseline evidence and deployed release unchanged: True
Saved report: docs\rental_ridge_validation.md
Saved outputs: data\interim\rental_ridge_validation


,model,n_forecasts,mae_aud_week,rmse_aud_week,mean_signed_error_aud_week
0,last_observation,589,10.9728,16.2303,-4.3582
1,ridge_lags_seasonal_alpha1,589,13.2135,18.6421,-1.9758
2,seasonal_naive,589,25.2937,38.9464,-13.0934


## Your checkpoint

Save this notebook with **Ctrl + S**. Send the final **RIDGE VALIDATION CHECKPOINT PASSED** output and the three-row error table, or the first error.

A passing checkpoint means that the method and data checks completed; it is not a promise that Ridge wins. Do not rerun notebook 09, change its dates, score the reserved final period, merge into `main`, or republish the app for this step.

**What you should be able to explain:** the model uses earlier rental values and a known calendar, learns only from the past at each origin, and is compared on exactly the same targets as the simple forecasts. A more complicated model may still forecast worse.
